In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [2]:
main_path = "./../cleaned_data/surveys/"

phq9 = pd.read_csv(f"{main_path}clean_PHQ-9.csv")

In [3]:
# Clean column names: remove commas, lowercase everything, replace spaces with underscores
phq9.columns = (
    phq9.columns.str.replace(",", "", regex=False)  # remove commas
    .str.replace(".", "", regex=False)  # remove periods
    .str.strip()  # remove leading/trailing whitespace
    .str.lower()  # lowercase
    .str.replace(" ", "_")  # replace spaces with underscores
)

for x in phq9.columns:
    print(x)

uid
type
little_interest_or_pleasure_in_doing_things
feeling_down_depressed_hopeless
trouble_falling_or_staying_asleep_or_sleeping_too_much
feeling_tired_or_having_little_energy
poor_appetite_or_overeating
feeling_bad_about_yourself_or_that_you_are_a_failure_or_have_let_yourself_or_your_family_down
trouble_concentrating_on_things_such_as_reading_the_newspaper_or_watching_television
moving_or_speaking_so_slowly_that_other_people_could_have_noticed_or_the_opposite_being_so_figety_or_restless_that_you_have_been_moving_around_a_lot_more_than_usual
thoughts_that_you_would_be_better_off_dead_or_of_hurting_yourself
response
total_score
depression_severity


In [4]:
print(phq9.head(2))

   uid type  little_interest_or_pleasure_in_doing_things  \
0  u00  pre                                            0   
1  u01  pre                                            1   

   feeling_down_depressed_hopeless  \
0                                1   
1                                1   

   trouble_falling_or_staying_asleep_or_sleeping_too_much  \
0                                                  0        
1                                                  1        

   feeling_tired_or_having_little_energy  poor_appetite_or_overeating  \
0                                      1                            0   
1                                      1                            0   

   feeling_bad_about_yourself_or_that_you_are_a_failure_or_have_let_yourself_or_your_family_down  \
0                                                  0                                               
1                                                  1                                               


In [5]:
# Segregating to pre and post into different files
phq9_pre = phq9[phq9["type"] == "pre"].copy()
phq9_post = phq9[phq9["type"] == "post"].copy()

# Drop the 'type' column safely
phq9_pre.drop("type", axis=1, inplace=True)
phq9_post.drop("type", axis=1, inplace=True)

# Add 'pre_' prefix to all columns except 'uid'
phq9_pre = phq9_pre.rename(
    columns={col: f"pre_{col}" for col in phq9_pre.columns if col != "uid"}
)

# Add 'post_' prefix to all columns except 'uid'
phq9_post = phq9_post.rename(
    columns={col: f"post_{col}" for col in phq9_post.columns if col != "uid"}
)

phq9_transformed = pd.merge(phq9_pre, phq9_post, on="uid", how="inner")

In [6]:
print(phq9_transformed.head(2))
print(phq9_post.head(2))
print(phq9_pre.head(2))

   uid  pre_little_interest_or_pleasure_in_doing_things  \
0  u00                                                0   
1  u01                                                1   

   pre_feeling_down_depressed_hopeless  \
0                                    1   
1                                    1   

   pre_trouble_falling_or_staying_asleep_or_sleeping_too_much  \
0                                                  0            
1                                                  1            

   pre_feeling_tired_or_having_little_energy  pre_poor_appetite_or_overeating  \
0                                          1                                0   
1                                          1                                0   

   pre_feeling_bad_about_yourself_or_that_you_are_a_failure_or_have_let_yourself_or_your_family_down  \
0                                                  0                                                   
1                                              

## Map labels

In [7]:
severity_map = {
    "Minimal depression": 0,
    "Mild depression": 1,
    "Moderate depression": 2,
    "Moderately severe depression": 3,
    "Severe depression": 4,
}

phq9_post["post_depression_class"] = phq9_post["post_depression_severity"].map(
    severity_map
)
phq9_pre["pre_depression_class"] = phq9_pre["pre_depression_severity"].map(severity_map)
phq9_transformed["pre_depression_class"] = phq9_transformed[
    "pre_depression_severity"
].map(severity_map)
phq9_transformed["post_depression_class"] = phq9_transformed[
    "post_depression_severity"
].map(severity_map)

In [8]:
display(phq9_post)
display(phq9_pre)
display(phq9_transformed)

,uid,post_little_interest_or_pleasure_in_doing_things,post_feeling_down_depressed_hopeless,post_trouble_falling_or_staying_asleep_or_sleeping_too_much,post_feeling_tired_or_having_little_energy,post_poor_appetite_or_overeating,post_feeling_bad_about_yourself_or_that_you_are_a_failure_or_have_let_yourself_or_your_family_down,post_trouble_concentrating_on_things_such_as_reading_the_newspaper_or_watching_television,post_moving_or_speaking_so_slowly_that_other_people_could_have_noticed_or_the_opposite_being_so_figety_or_restless_that_you_have_been_moving_around_a_lot_more_than_usual,post_thoughts_that_you_would_be_better_off_dead_or_of_hurting_yourself,post_response,post_total_score,post_depression_severity,post_depression_class
46,u00,0,0,0,1,0,1,1,0,0,1,3,Minimal depression,0.0
47,u01,0,1,1,1,0,0,0,1,0,1,4,Minimal depression,0.0
48,u02,2,0,0,1,1,0,1,0,0,1,5,Mild depression,1.0
49,u03,1,1,0,1,0,1,0,0,0,1,4,Minimal depression,0.0
50,u04,1,1,1,1,1,2,1,0,0,1,8,Mild depression,1.0
51,u05,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN
52,u07,1,1,2,2,2,0,0,0,0,1,8,Mild depression,1.0
53,u09,0,0,0,0,1,1,0,0,0,0,2,Minimal depression,0.0
54,u10,0,0,0,1,1,0,0,2,0,0,4,Minimal depression,0.0
55,u14,0,0,1,1,1,0,0,0,0,-1,3,Minimal depression,0.0


,uid,pre_little_interest_or_pleasure_in_doing_things,pre_feeling_down_depressed_hopeless,pre_trouble_falling_or_staying_asleep_or_sleeping_too_much,pre_feeling_tired_or_having_little_energy,pre_poor_appetite_or_overeating,pre_feeling_bad_about_yourself_or_that_you_are_a_failure_or_have_let_yourself_or_your_family_down,pre_trouble_concentrating_on_things_such_as_reading_the_newspaper_or_watching_television,pre_moving_or_speaking_so_slowly_that_other_people_could_have_noticed_or_the_opposite_being_so_figety_or_restless_that_you_have_been_moving_around_a_lot_more_than_usual,pre_thoughts_that_you_would_be_better_off_dead_or_of_hurting_yourself,pre_response,pre_total_score,pre_depression_severity,pre_depression_class
0,u00,0,1,0,1,0,0,0,0,0,0,2,Minimal depression,0.0
1,u01,1,1,1,1,0,1,0,0,0,2,5,Mild depression,1.0
2,u02,2,1,2,2,2,1,1,2,0,1,13,Moderate depression,2.0
3,u03,0,1,0,0,0,0,0,1,0,1,2,Minimal depression,0.0
4,u04,1,1,0,1,1,1,1,0,0,1,6,Mild depression,1.0
5,u05,0,0,1,0,1,0,0,0,0,-1,2,Minimal depression,0.0
6,u07,1,1,0,1,2,1,1,0,0,0,7,Mild depression,1.0
7,u08,1,1,0,0,1,1,1,0,0,0,5,Mild depression,1.0
8,u09,0,0,1,1,1,0,1,0,0,0,4,Minimal depression,0.0
9,u10,0,0,0,0,0,0,0,0,0,-1,0,NaN,NaN


,uid,pre_little_interest_or_pleasure_in_doing_things,pre_feeling_down_depressed_hopeless,pre_trouble_falling_or_staying_asleep_or_sleeping_too_much,pre_feeling_tired_or_having_little_energy,pre_poor_appetite_or_overeating,pre_feeling_bad_about_yourself_or_that_you_are_a_failure_or_have_let_yourself_or_your_family_down,pre_trouble_concentrating_on_things_such_as_reading_the_newspaper_or_watching_television,pre_moving_or_speaking_so_slowly_that_other_people_could_have_noticed_or_the_opposite_being_so_figety_or_restless_that_you_have_been_moving_around_a_lot_more_than_usual,pre_thoughts_that_you_would_be_better_off_dead_or_of_hurting_yourself,pre_response,pre_total_score,pre_depression_severity,post_little_interest_or_pleasure_in_doing_things,post_feeling_down_depressed_hopeless,post_trouble_falling_or_staying_asleep_or_sleeping_too_much,post_feeling_tired_or_having_little_energy,post_poor_appetite_or_overeating,post_feeling_bad_about_yourself_or_that_you_are_a_failure_or_have_let_yourself_or_your_family_down,post_trouble_concentrating_on_things_such_as_reading_the_newspaper_or_watching_television,post_moving_or_speaking_so_slowly_that_other_people_could_have_noticed_or_the_opposite_being_so_figety_or_restless_that_you_have_been_moving_around_a_lot_more_than_usual,post_thoughts_that_you_would_be_better_off_dead_or_of_hurting_yourself,post_response,post_total_score,post_depression_severity,pre_depression_class,post_depression_class
0,u00,0,1,0,1,0,0,0,0,0,0,2,Minimal depression,0,0,0,1,0,1,1,0,0,1,3,Minimal depression,0.0,0.0
1,u01,1,1,1,1,0,1,0,0,0,2,5,Mild depression,0,1,1,1,0,0,0,1,0,1,4,Minimal depression,1.0,0.0
2,u02,2,1,2,2,2,1,1,2,0,1,13,Moderate depression,2,0,0,1,1,0,1,0,0,1,5,Mild depression,2.0,1.0
3,u03,0,1,0,0,0,0,0,1,0,1,2,Minimal depression,1,1,0,1,0,1,0,0,0,1,4,Minimal depression,0.0,0.0
4,u04,1,1,0,1,1,1,1,0,0,1,6,Mild depression,1,1,1,1,1,2,1,0,0,1,8,Mild depression,1.0,1.0
5,u05,0,0,1,0,1,0,0,0,0,-1,2,Minimal depression,0,0,0,0,0,0,0,0,0,0,0,NaN,0.0,NaN
6,u07,1,1,0,1,2,1,1,0,0,0,7,Mild depression,1,1,2,2,2,0,0,0,0,1,8,Mild depression,1.0,1.0
7,u09,0,0,1,1,1,0,1,0,0,0,4,Minimal depression,0,0,0,0,1,1,0,0,0,0,2,Minimal depression,0.0,0.0
8,u10,0,0,0,0,0,0,0,0,0,-1,0,NaN,0,0,0,1,1,0,0,2,0,0,4,Minimal depression,NaN,0.0
9,u14,0,0,0,1,0,0,0,0,0,0,1,Minimal depression,0,0,1,1,1,0,0,0,0,-1,3,Minimal depression,0.0,0.0


In [9]:
phq9_transformed.to_csv("./lstm_process_data/phq9_transformed_full.csv", index=False)
phq9_pre.to_csv("./lstm_process_data/phq9_transformed_pre.csv", index=False)
phq9_post.to_csv("./lstm_process_data/phq9_transformed_post.csv", index=False)